#  Tool Calling: Travel Assistant

what's the weather in Rome *right now*? If you asked an LLM, it would have to
guess, admit it has no idea or even worst, invent data. LLMs are trained on historical data, they have a
**knowledge cutoff** and they dont have access onto the live world. That's a real limitation for applications that require current or even real time data. 


For that reason we're going to hand an LLM a set of **tools**, Python functions it can call when it needs, and watch it reach outside itself to get the real answer.

By the end, you'll have a working **travel assistant** that can:
- 🌤️ Check the current weather in a city
- 🕐 Look up the local time at a destination
- 💱 Convert currencies
- Use a real search engine to complement LLM responses. 

...and you'll have built it using **LangChain**, one of the most widely used frameworks for this
kind of work.

## So what does LangChain actually do?

Using an LLM provider's (OpenAI, Anthropic, Gemini, and plenty more)raw API directly works, but it comes with a lot of work to identify and use each provider structure. You end up writing JSON tool schemas by hand, learning the differences of each
provider's response format, and rebuilding the same "call model → run tools → feed results back →
repeat" loop.

LangChain sits on top of those provider APIs and
hands you:

- **One interface, many providers.** A common `ChatModel` interface, so changing Gemini for
  another provider is often a one line change.
- **A standard message format.** `HumanMessage`, `AIMessage`, `ToolMessage`, etc. give every
  conversation turn a consistent shape.
- **Tool schemas generated from your code.** You'll see this one in action in a moment: the
  `@tool` decorator reads a function's signature and docstring and writes the schema for you.
- **Higher-level orchestration** (like `create_agent`) that can run the entire tool-calling loop
  for you.

you'll build the loop LLM-tool by hand first, so you can see the fundamentals and then you will see the automated
version.

⚠️ Only if you are running the code outside of Github Codespace or Instruqt, uncomment the
following cell and run it.
This reads the `.env` file in your project and transforms its content into env variables.
This way you don't have to hard code your secrets in the code.

In [ ]:
#from dotenv import load_dotenv
#load_dotenv()

## Setup

We'll use `langchain-litellm`, LangChain's integration for LiteLLM, slecting the model
`"gemini/gemini-3.5-flash-lite"`.



In [ ]:
from langchain_litellm import ChatLiteLLM

model = ChatLiteLLM(model="gemini/gemini-3.5-flash-lite")

## Without tools

An LLM alone can not interact/act with the outside world, let's prove the point before you implement the tools. We'll ask a travel question that requires live information: current weather, local time, and exchange rates. Watch what the model does with it,
it'll guess, or confidently state something outdated. Run the cell below and see for
yourself.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

system_prompt = """
You are a helpful travel assistant.
"""

user_query = """
I'm flying to Rome tomorrow. Can you tell me:
1. What is the current weather there?
2. What time is it in Rome right now?
3. How much is 500 USD in Euros?
"""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_query),
]

response = model.invoke(messages)

print(response.content)

The answer is pretty underwhelming. 
 
Notice that `response` here is an `AIMessage` object rather than
a raw dict, LangChain wraps every provider's response in the same set of message classes, which is
part of what makes the interface consistent across models.
Let's actually fix the wrong-answer part.

## 🌤️ With a Weather Tool

The model can't check the weather, but a Python function can (or at least mock one until you wire
up a real API).

Before we use LangChain to define the functions for the model, let's see what it takes to describe a tool to a model the
*hard* way: writing the JSON schema by hand. Then we'll let `@tool` do it for us, and
you'll be able to feel the difference immediately.

We'll define a get_weather function and describe it to the model using a tool schema so it knows:

- what function is called
- what it does
- what arguments it expects

In [ ]:
# This is what you'd have to write, BY HAND, to describe a single tool to a model directly,
#
def get_weather(city: str) -> dict:
    """Get the current weather for a city.
    Note: uses mock data, replace with a real API (e.g. OpenWeatherMap) in production.
    """
    weather_data = {
        "rome": {"temperature_c": 22, "condition": "Partly cloudy", "humidity_pct": 65},
        "paris": {"temperature_c": 18, "condition": "Light rain", "humidity_pct": 82},
        "new york": {"temperature_c": 15, "condition": "Sunny", "humidity_pct": 40},
        "london": {"temperature_c": 12, "condition": "Overcast", "humidity_pct": 75},
        "sydney": {"temperature_c": 28, "condition": "Clear", "humidity_pct": 55},
        "tokyo": {"temperature_c": 17, "condition": "Partly cloudy", "humidity_pct": 70},
        "dubai": {"temperature_c": 35, "condition": "Sunny", "humidity_pct": 30},
    }
    data = weather_data.get(city.lower(), {"temperature_c": 20, "condition": "Clear", "humidity_pct": 50})
    result = {"city": city, **data}
    print("~" * 20, f"get_weather({city!r}) → {result}", "~" * 20)
    return result


weather_tool_schema_manual = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city (e.g. 'Rome', 'Paris').",
                }
            },
            "required": ["city"],
        },
    },
}


import json
print(json.dumps(weather_tool_schema_manual, indent=2))

That's a lot of code just for one function with one argument. Now imagine your `get_weather`
function evolves, say you add a `units` parameter for Celsius vs. Fahrenheit. You'd need to
remember to update this dict too, by hand, every single time, for every single tool. 

Let's see the alternative.

## Using `@tool` decorator

We'll define the exact same `get_weather` function, but this time using LangChain. The `@tool` decorator inspects the function's **type hints** and **docstring** and
builds the schema automatically, no hand-written JSON required. Keep the docstring accurate: the
model reads it to decide when to call the tool, and LangChain uses it as the `description` field.

> **Training note:** The tools in this notebook use mock data and simple fallback values so the
> exercise keeps running even for unsupported inputs. In production, a tool should return an
> explicit unknown/unsupported result instead of plausible default weather or currency data.

In [ ]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> dict:
    """Get the current weather for a city.


    Args:
        city: The name of the city (e.g. 'Rome', 'Paris').
    """
    weather_data = {
        "rome": {"temperature_c": 22, "condition": "Partly cloudy", "humidity_pct": 65},
        "paris": {"temperature_c": 18, "condition": "Light rain", "humidity_pct": 82},
        "new york": {"temperature_c": 15, "condition": "Sunny", "humidity_pct": 40},
        "london": {"temperature_c": 12, "condition": "Overcast", "humidity_pct": 75},
        "sydney": {"temperature_c": 28, "condition": "Clear", "humidity_pct": 55},
        "tokyo": {"temperature_c": 17, "condition": "Partly cloudy", "humidity_pct": 70},
        "dubai": {"temperature_c": 35, "condition": "Sunny", "humidity_pct": 30},
    }
    data = weather_data.get(city.lower(), {"temperature_c": 20, "condition": "Clear", "humidity_pct": 50})
    result = {"city": city, **data}
    print("~" * 20, f"get_weather({city!r}) -> {result}", "~" * 20)
    return result


# This inspect what the decorator generated for us, this is the schema the model actually sees:
print("name:", get_weather.name)
print("description:", get_weather.description)
print("args schema:", get_weather.args)

Run the cell above, and compare what gets printed to the 18-line JSON dict a couple of cellls up when calling the tool the hard way. Same
informationn, and it came straight out of the function's signature and docstring.
Add a parameter, update a docstring, and the schema updates itself. 

## Binding tools to the model

`bind_tools()` returns a new model object that knows about the tools you give it, so the model can
choose to call them on its own.

In [ ]:
system_prompt = """
You are a helpful travel assistant.
Use the available tools to fetch current information. Never rely on your own knowledge for weather, time, or exchange rates.
"""

user_query = """
I'm flying to Rome tomorrow. What's the weather like there right now?
"""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_query),
]

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(messages)
print(response.content)
print(response.tool_calls)

`response.content` is empty, the model wants to call a tool instead of answering directly.
LangChain normalizes tool calls into a `.tool_calls` attribute (a list of dicts with `name`, `args`,
and `id`) directly on the message, for every provider, so you don't need to use a
provider-specific response shape to find them.

## Executing the tool and feeding the result back

Now we run the tool the model asked for, then send the result back as part of the conversation so
the model can turn it into a natural language answer. LangChain uses a `ToolMessage` object for
this, and `tool_call_id` links it back to the specific call that requested it.

In [ ]:
from langchain_core.messages import ToolMessage

# Append the model's tool call request to the conversation.
messages.append(response)

for tool_call in response.tool_calls:
    # Tools defined with @tool can be invoked directly with the args dict the model produced.
    result = get_weather.invoke(tool_call["args"])
    messages.append(
        ToolMessage(content=str(result), tool_call_id=tool_call["id"])
    )

final_response = model_with_tools.invoke(messages)
print(final_response.content)

## Adding more tools

Let's add local time and currency conversion, defined exactly the same
way with `@tool`, this follow the same structure as in the get_weather. 

In [ ]:
from datetime import datetime, timedelta


@tool
def get_current_time(city: str) -> dict:
    """Get the current local time for a city.

    Args:
        city: The name of the city.
    """
    utc_offsets = {
        "rome": 1,
        "paris": 1,
        "new york": -4,
        "london": 0,
        "sydney": 10,
        "tokyo": 9,
        "dubai": 4,
    }
    offset = utc_offsets.get(city.lower(), 0)
    local_time = datetime.utcnow() + timedelta(hours=offset)
    result = {"city": city, "local_time": local_time.strftime("%Y-%m-%d %H:%M"), "utc_offset": offset}
    print("~" * 20, f"get_current_time({city!r}) -> {result}", "~" * 20)
    return result


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert.
        from_currency: Source currency code (e.g. 'USD', 'EUR').
        to_currency: Target currency code (e.g. 'JPY', 'GBP').
    """
    rates_to_usd = {
        "usd": 1.0,
        "eur": 1.08,
        "gbp": 1.27,
        "jpy": 0.0067,
        "aud": 0.66,
    }
    from_rate = rates_to_usd.get(from_currency.lower(), 1.0)
    to_rate = rates_to_usd.get(to_currency.lower(), 1.0)
    converted = amount * from_rate / to_rate
    result = {
        "amount": amount,
        "from_currency": from_currency.upper(),
        "to_currency": to_currency.upper(),
        "converted_amount": round(converted, 2),
    }
    print("~" * 20, f"convert_currency({amount!r}, {from_currency!r}, {to_currency!r}) -> {result}", "~" * 20)
    return result


tools = [get_weather, get_current_time, convert_currency]
model_with_tools = model.bind_tools(tools)

## Building the agent loop

A real question ("weather, time, *and* currency, please") needs more
than one tool call. So we need a loop, keep invoking the model, execute whatever tools it asks
for, feed the results back as `ToolMessage`s, and stop once the model responds with plain content
instead of more tool calls. 

Run the following cell and watch it work: the model calls each tool in turn and assembles a complete, accurate
answer. Notice the `~~~` lines, those are the print statements
showing exactly which tools got invoked and with what arguments.


In [ ]:
# Map tool names to the LangChain tool objects our runtime can execute.
tool_registry = {t.name: t for t in tools}


def run_travel_assistant(messages, model_with_tools, tool_registry):
    while True:
        response = model_with_tools.invoke(messages)
        messages.append(response)

        if response.content:
            print(response.content)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            if tool_name in tool_registry:
                result = tool_registry[tool_name].invoke(tool_call["args"])
                messages.append(
                    ToolMessage(content=str(result), tool_call_id=tool_call["id"])
                )


system_prompt = """
You are a helpful travel assistant.
Use the available tools to fetch current information. Never rely on your own knowledge for weather, time, or exchange rates.
"""

user_query = """
I'm flying to Tokyo tomorrow. Can you tell me:
1. What is the current weather there?
2. What time is it in Tokyo right now?
3. How much is 500 USD in Japanese Yen?
"""

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_query),
]

run_travel_assistant(messages, model_with_tools, tool_registry)

# Create_Agent

Langchain come with a lot of abstractions that can be used to simplify the procces of building an agent. For example,  `create_agent`  wraps the previous loop so you don't write it yourself, and adds things
like configurable max iterations, built-in logging/tracing, and easy swapping in of memory. https://reference.langchain.com/python/langchain/agents/factory/create_agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    debug=True,
)

response = agent.invoke({"messages": [{"role": "user", "content": user_query}]})
print(response["messages"][-1].content)

In this case you dont nee to use `tool_registry` dict, no `while True`, no manual `ToolMessage`
wrapping. `create_agent` invocation runs that whole loop internally, and `debug=True` prints each step
(tool called, arguments, result) as it happens, so you still get visibility into what's going on.


Now it's your turn to add a tool from scratch 

## Add a Local Highlights Tool (with a *real* search engine)

Every tool so far has returned mock data, reliable for learning, but not impressive. Let's
change that: this tool will search the live web using **DuckDuckGo**,a search engine. 




It is important to notice that a search engine doesn't hand back a dictionary like `{"top_sight": "Colosseum"}`.
It hands back messy, real paragraphs of text. That's actually the point, you'll see the model
read those snippets and summarize them into a clean answer on its own, which is a much more
realistic picture of what tool calling looks like in production.

> ⚠️ **This tool needs live internet access.** If you're running in a sandboxed or offline
> environment, it may fail where the mock tools above never would, that's expected, not a bug in
> your code.

**Your task:** build `get_local_highlights(city)`, a tool that searches the web for top attractions,
must-try local food, and cultural tips for a destination.

**Steps:**
1. Build a good search query inside `get_local_highlights` (think: what would *you* type into a
   search engine to find out what's worth doing and eating somewhere?) Hint: use f-string to use the tool input. 
3. Add the tool to the local `tools` list and create a new prompt considering the new tool.
4. re define the agent using create agent, call it `agent_with_highlights`.


Give it a try yourself before scrolling to the solution at the end of the notebook!

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()


@tool
def get_local_highlights(city: str) -> str:
    """Search the web for top attractions, local food, and cultural tips for a city.

    Use this when the user wants recommendations on what to see, eat, or do at a destination.

    Args:
        city: The name of the city (e.g. 'Tokyo', 'Rome').
    """
    #TO-DO
    #query = .....
    result = search_tool.invoke(query)
    print("~" * 20, f"get_local_highlights({city!r}) -> searched: {query!r}", "~" * 20)
    return result

#TO-DO
#tools_with_highlights = ....

#TO-DO
#highlights_system_prompt = """   """

#TO-DO
#Remember that now we now how to use create_agent

#agent_with_highlights = create_agent()

test_query = """
I'm planning a trip to Japan next month.
What's the weather in Tokyo, what time is it there now, how much is 300 EUR in Yen,
and what are some local highlights, food and sights, I shouldn't miss in Tokyo?
"""

# The followign loop stream the result to see each step:

printed = 0
for step in agent_with_highlights.stream(
    {"messages": [{"role": "user", "content": test_query}]},
    stream_mode="values",
):
    for msg in step["messages"][printed:]:
        msg.pretty_print()
    printed = len(step["messages"])

---

##  Solution

Here's a complete, working implementation of
`get_local_highlights`.

In [ ]:
# %load ../../answers/c01_n01_tool_calling_LC.py